In [ ]:
import asyncio
import re
import json
import ast
import sys
import os
import getpass
from typing import TypedDict, Annotated, Any

from langchain_together import ChatTogether
from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient

import os
import getpass
from langchain_fireworks import ChatFireworks

if "FIREWORKS_API_KEY" not in os.environ:
    os.environ["FIREWORKS_API_KEY"] = getpass.getpass("Enter your Fireworks API key: ")

In [ ]:
class AgentState(TypedDict):
    messages: list
    step: str
    query: str
    intent: str
    models: list
    domains: list
    subdomains: dict
    context: str
    metrics: dict
    selected_llm: Any

class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        self.model = model.bind_tools(tools)
        self.tools = {t.name: t for t in tools}
        self.metric_weights = {
            "accuracy": 0.7,
            "latency_ms": 0.2,
            "memory_mb": 0.1,
        }
        self.graph = self._build_graph(checkpointer)

    def _build_graph(self, cp):
        g = StateGraph(AgentState)
        g.add_node("init", self.init_step)
        g.add_node("collect_metrics", self.collect_metrics_step)
        g.add_node("respond", self.respond_step)
        g.add_edge("init", "collect_metrics")
        g.add_edge("collect_metrics", "respond")
        g.set_entry_point("init")
        return g.compile(checkpointer=cp)

    def classify_intent(self, query) -> str:
        print("DEBUG: helper function 'classify_intent'")
        if isinstance(query, list):
            q = " ".join(map(str, query)).lower()
        else:
            q = str(query).lower()

        print(f"DEBUG: classify_intent received query: '{q}'")
        
        if re.search(r"\b(compare|better|best|prefer|efficient|memory|performance)\b", q):
            return "compare"
        # 2. Check for 'metrics' next
        if re.search(r"\b(metrics)\b", q):
            return "metrics"
        # 3. Fallback to just listing models
        if re.search(r"\b(models|list|available)\b", q):
            return "list_models"
            
        return "metrics"

    async def map_query_to_domain_subdomain_llm(self, query, domains, subdomains):
        print("DEBUG: helper function 'map_query_to_domain_subdomain_llm'")
        lines = []
        for d in domains:
            subs = [str(s) for s in subdomains.get(d, [])] or ["None"]
            lines.append(f"- {d}: {', '.join(subs)}")
            
        prompt = f"""
        You are an AI routing assistant. Map the user's query to the SINGLE most relevant domain and subdomain from the list below.
        
        Available domains and subdomains:
        {chr(10).join(lines)}
        
        User query: "{query}"
        
        Respond EXACTLY in this format with no extra text or markdown:
        Domain: <exact domain name from list or None>
        Subdomain: <exact subdomain name from list or None>
        """
        resp = await self.model.ainvoke([HumanMessage(content=prompt)])
        content = resp.content.strip()
        print(f"DEBUG: Raw LLM Routing Output:\n{content}")
        
        dm = re.search(r"Domain:\s*(.*)", content, re.IGNORECASE)
        sd = re.search(r"Subdomain:\s*(.*)", content, re.IGNORECASE)
        
        domain = dm.group(1).strip() if dm and dm.group(1).strip().lower() != 'none' else None
        sub = sd.group(1).strip() if sd and sd.group(1).strip().lower() != 'none' else None
        
        print(f"DEBUG: mapping result -> domain={domain}, subdomain={sub}")
        return domain, sub, f"LLM mapped to {domain}/{sub} or 'None'"

    def normalize_accuracy(self, v):
        return v / 100 if isinstance(v, (int, float)) and v > 1 else v

    def normalize_metric(self, value, metric_type, all_values):
        if not all_values:
            return 0.0
        min_val = min(all_values)
        max_val = max(all_values)
        if max_val == min_val:
            return 1.0 
        if metric_type in ["latency_ms", "memory_mb"]:
            return (max_val - value) / (max_val - min_val)
        return (value - min_val) / (max_val - min_val)

    def select_best_llm(self, metrics, key):
        print("DEBUG: helper function 'select_best_llm'")
        if not metrics:
            return None, "No metrics data available"

        metric_values = {"accuracy": [], "latency_ms": [], "memory_mb": []}
        for model_key, data in metrics.items():
            if isinstance(data, dict):
                for metric in metric_values:
                    value = data.get(metric, 0.0)
                    if isinstance(value, (int, float)):
                        metric_values[metric].append(value)

        scores = {}
        for model_key, data in metrics.items():
            if not isinstance(data, dict):
                continue
            score = 0.0
            for metric, weight in self.metric_weights.items():
                value = data.get(metric, 0.0)
                if isinstance(value, (int, float)) and metric_values[metric]:
                    normalized = self.normalize_metric(value, metric, metric_values[metric])
                    score += weight * normalized
            scores[model_key] = score
            print(f"DEBUG: weighted score: {score:.3f} for {model_key}")

        if not scores:
            return None, "No valid metrics for comparison"

        best_model = max(scores.items(), key=lambda x: x[1])[0]
        return best_model, f"Selected {best_model} based on weighted metrics"

    async def init_step(self, state: AgentState):
        print("DEBUG: Entering node INIT")
        query = state["messages"][-1].content
        intent = self.classify_intent(query)
        print("DEBUG: fetching models/domains/subdomains")
        
        def parse_mcp_list(raw_output):
            
            if isinstance(raw_output, list) and len(raw_output) > 0 and isinstance(raw_output[0], dict):
                return [item.get("text", "") for item in raw_output if "text" in item]
            
            if isinstance(raw_output, str):
                try: return json.loads(raw_output)
                except: return []
                
            if isinstance(raw_output, list) and all(isinstance(x, str) for x in raw_output):
                return raw_output
                
            return []

        raw_models = await self.tools["list_models"].ainvoke({})
        models = parse_mcp_list(raw_models)

        raw_domains = await self.tools["list_domains"].ainvoke({})
        domains = parse_mcp_list(raw_domains)

        subdomains = {}
        for d in domains:
            raw_sub = await self.tools["list_sub_domains"].ainvoke({"domain": d})
            subdomains[d] = parse_mcp_list(raw_sub)
            
        return {
            "step": "init",
            "query": query,
            "intent": intent,
            "models": models,
            "domains": domains,
            "subdomains": subdomains,
            "context": "",
            "metrics": {},
            "selected_llm": None,
            "messages": []
        }
        
    async def collect_metrics_step(self, state: AgentState):
        print("DEBUG: Entering node collect_metrics_step")
        query = state["query"]
        intent = state["intent"]
        metrics = {}
        messages = []
    
        domain, subdomain, mapping_reason = await self.map_query_to_domain_subdomain_llm(
            query, state["domains"], state["subdomains"]
        )
        
        messages.append(ToolMessage(tool_call_id="mapping", name="mapping", content=mapping_reason))
    
        def parse_tool_result(raw):
            try:
                if isinstance(raw, dict): data = raw
                elif isinstance(raw, str): data = json.loads(raw)
                else: data = {}
            except Exception:
                try: data = ast.literal_eval(raw)
                except Exception: data = {}
    
            if isinstance(data, dict) and 'accuracy' in data:
                data['accuracy'] = self.normalize_accuracy(data['accuracy'])
            return data

        if intent in ["metrics", "compare"] and domain:
            api_domain = domain.title()
            api_sub = subdomain.lower() if subdomain else None
    
            for model_name in state["models"]:
                print(f"DEBUG: Calling metrics for {model_name} in {api_domain}/{api_sub}")
                try:
                    if api_sub:
                        raw = await self.tools["get_metric_domain_subdomain"].ainvoke({
                            "model": model_name, "domain": api_domain, "sub_domain": api_sub
                        })
                    else:
                        raw = await self.tools["get_metric_domain"].ainvoke({
                            "model": model_name, "domain": api_domain
                        })
    
                    if isinstance(raw, list) and len(raw) > 0 and isinstance(raw[0], dict):
                        raw_text = raw[0].get("text", "{}")
                    else:
                        raw_text = str(raw)

                    data = parse_tool_result(raw_text)
                    if data:
                        key = f"{model_name}_{api_domain}_{api_sub}" if api_sub else f"{model_name}_{api_domain}"
                        metrics[key] = data
                        messages.append(ToolMessage(
                            tool_call_id=f"metrics_{key}",
                            name=("get_metric_domain_subdomain" if api_sub else "get_metric_domain"),
                            content=str(data)
                        ))
                except Exception as e:
                    print(f"Error fetching metrics for {model_name}: {e}")
            try:
                best_model, reason = self.select_best_llm(metrics, api_sub or api_domain)
                print(f"DEBUG: Best model: {best_model}\nReason: {reason}")
                state["selected_llm"] = best_model
                state["context"] = reason
            except Exception as e:
                print(f"Error selecting best model: {e}")
                state["selected_llm"] = None
                state["context"] = "Error during model selection."
    
        return {
            "step": "metrics",
            "metrics": metrics,
            "messages": messages,
            "selected_llm": state.get("selected_llm"),
            "context": state.get("context")
        }

    async def respond_step(self, state: AgentState):
        print("DEBUG: Entering node respond_step\n")
        query = str(state.get("query") or "").lower()
        intent = state.get("intent")
        metrics = state.get("metrics", {})
        context = state.get("context", "")
        selected_llm = state.get("selected_llm")
        
        response = f"Based on the analysis and context:\nContext: {context}\n\n"
        if intent == "list_models" and state.get("models"):
            response += f"Available models: {', '.join(state['models'])}\n"
        elif intent in ["compare", "metrics"]:
            if intent == "compare" and metrics.get("comparison"):
                comp = metrics["comparison"]
                best, best_acc = max(comp.items(), key=lambda kv: kv[1])
                response += f"Best model: {best} (accuracy={best_acc:.2f})\n"
                response += "Model accuracies:\n" + "\n".join(f"{m}: {a:.2f}" for m, a in comp.items()) + "\n"
            elif intent == "metrics" and metrics:
                response += "Metrics collected for models:\n"
                for m, v in metrics.items():
                    response += f"{m}:\n"
                    for metric_name, value in v.items():
                        response += f"  {metric_name}: {value}\n"
                    response += "\n"
            else:
                response += "No data found for this query\n"
            
        if selected_llm:
            model_parts = selected_llm.split("_")
            model_name = "_".join(model_parts[:2])
            response += f"\nRecommended Model for routing: {model_name}"
            
        return {"messages": [HumanMessage(content=response)]}



In [7]:
router_prompt = """
You are an Intelligent router responsible for handling queries within a Multi-Agent System (MAS) framework.

Upon receiving a query from the MAS, your task is to:
1. Accurately classify the query's intent.
2. Identify the relevant domain(s) and subdomain(s) associated with the query.
3. Utilize available tool calls through the MCP server to:
   - Retrieve necessary information such as available models, domains, and subdomains.
   - Fetch and normalize model performance metrics.
   - Select the most suitable LLM(s) capable of addressing the query effectively.
4. If the query spans multiple domains or subdomains, you are allowed to make multiple tool calls to ensure optimal model selection.
5. Your response must only contain the selected model's name in the following format: "Model:___"
"""

mcp_client = MultiServerMCPClient({
    "csv_r": {
        "command": sys.executable,
        "args": ["-u", "main.py"],
        "cwd": "./mcp_server",
        "transport": "stdio",
    }
})

memory = MemorySaver()
async def main():
    print("Fetching tools from MCP server...")
    tools = await mcp_client.get_tools()
    

    model = ChatFireworks(
        model="accounts/fireworks/models/llama-v3p3-70b-instruct", 
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2
    )
    
    agent = Agent(model, tools, checkpointer=memory, system=router_prompt)

    query_text = "Give the metrics for History for al models"
    thread = {"configurable": {"thread_id": "4"}}
    
    state = {
        "messages": [HumanMessage(content=query_text)],
        "step": "init",
        "models": [],
        "domains": [],
        "subdomains": {},
        "query": query_text,
        "intent": "",
        "metrics": {},
        "context": "",
        "selected_llm": None
    }
    
    print(f"\nInvoking Graph with query: '{query_text}'")
    result = await agent.graph.ainvoke(state, thread)
    
    print("\n--- FINAL OUTPUT ---")
    print(result["messages"][-1].content)

await main()

Fetching tools from MCP server...


Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7f18b0615dc0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7f18b0616480>



Invoking Graph with query: 'Give the metrics for History for al models'
DEBUG: Entering node INIT
DEBUG: helper function 'classify_intent'
DEBUG: classify_intent received query: 'give the metrics for history for al models'
DEBUG: fetching models/domains/subdomains
DEBUG: Entering node collect_metrics_step
DEBUG: helper function 'map_query_to_domain_subdomain_llm'
DEBUG: Raw LLM Routing Output:
Domain: General-Knowledge
Subdomain: history
DEBUG: mapping result -> domain=General-Knowledge, subdomain=history
DEBUG: Calling metrics for Llama3.2_1B in General-Knowledge/history
DEBUG: Calling metrics for Qwen2.5_1.5B in General-Knowledge/history
DEBUG: helper function 'select_best_llm'
DEBUG: weighted score: 0.300 for Llama3.2_1B_General-Knowledge_history
DEBUG: weighted score: 0.700 for Qwen2.5_1.5B_General-Knowledge_history
DEBUG: Best model: Qwen2.5_1.5B_General-Knowledge_history
Reason: Selected Qwen2.5_1.5B_General-Knowledge_history based on weighted metrics
DEBUG: Entering node respon